In [11]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join('..', 'geoai', 'utils_ml')))
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import segmentation_models_pytorch as smp 
from torch.utils.data import Dataset
from dl_ops import SegmentationDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)


cpu


In [12]:
# set hyperparameters
EPOCHS = 1
BS = 2

In [14]:
train_ds = SegmentationDataset(path_name='train')
train_dataloader = DataLoader(train_ds, batch_size=BS, shuffle=True)
val_ds = SegmentationDataset(path_name='val')
val_dataloader = DataLoader(val_ds, batch_size=BS, shuffle=True)


In [15]:
# Use model using the FPN (Feature Pyramid Network) architecture from the
# segmentation_models_pytorch (smp) library

model = smp.FPN(
    encoder_name="se_resnext50_32x4d",
    encoder_weights=None,
    classes=6,
    activation="sigmoid",
)

# Load the weights
model.encoder.load_state_dict(
    torch.load("../trained_models/se_resnext50_32x4d-a260b3a4.pth")
)
# download the weights if SSL error: http://data.lip6.fr/cadene/pretrainedmodels/se_resnext50_32x4d-a260b3a4.pth

C:\Users\Reginald\AppData\Local\Temp\ipykernel_952\2216620066.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("../trained_models/se_resnext50_32x4d-a260b3a4.

In [16]:
# Define the loss function and optimizer
optimizer = torch.optim.Adam([ 
    dict(params=model.parameters(), lr=0.0001),
])
criterion = criterion = nn.CrossEntropyLoss()

In [17]:
# Train the model
train_losses, val_losses = [], []
for e in range(EPOCHS):
    model.train()
    running_train_loss, running_val_loss = 0, 0
    for i, data in enumerate(train_dataloader):
        #training phase
        image_i, mask_i = data
        image = image_i.to(DEVICE)
        mask = mask_i.to(DEVICE)
        
        # reset gradients
        optimizer.zero_grad() 
        #forward
        output = model(image.float())
        
        # calc losses
        train_loss = criterion(output.float(), mask.long())

        # back propagation
        train_loss.backward()
        optimizer.step() #update weight          
        
        running_train_loss += train_loss.item()
    train_losses.append(running_train_loss) 
    
    # validation
    model.eval()
    with torch.no_grad():
        for i, data in enumerate(val_dataloader):
            image_i, mask_i = data
            image = image_i.to(DEVICE)
            mask = mask_i.to(DEVICE)
            #forward
            output = model(image.float())
            # calc losses
            val_loss = criterion(output.float(), mask.long())
            running_val_loss += val_loss.item()
    val_losses.append(running_val_loss) 
    
        
    print(f"Epoch: {e}: Train Loss: {np.median(running_train_loss)}, Val Loss: {np.median(running_val_loss)}")

Epoch: 0: Train Loss: 430.85863161087036, Val Loss: 22.820506811141968


In [22]:
# %% save model
torch.save(model.state_dict(), f'trained_models/FPN_epochs_{EPOCHS}_crossentropy_state_dict.pth')